# 07 — Treinamento de Modelos

Treinamento de **5 modelos** de classificacao para predicao de First Payment Default (FPD)
com otimizacao de hiperparametros via Optuna.

| # | Modelo | Tipo | HPO |
|---|--------|------|-----|
| 1 | LightGBM v2 | Gradient Boosting | Optuna (50 trials) |
| 2 | XGBoost | Gradient Boosting | Optuna (50 trials) |
| 3 | CatBoost | Gradient Boosting | Optuna (50 trials) |
| 4 | Random Forest | Ensemble | Default |
| 5 | Logistic Regression L1 v2 | Linear | Default |

**Split temporal**: Train (202410-202501), OOS (202501), OOT (202502-202503)

**Metricas**: KS, AUC, Gini, PSI por split

In [ ]:
# ---------------------------------------------------------------------------
# Threading config + imports
# ---------------------------------------------------------------------------
import os

# Configurar threads ANTES de importar libs numericas
NCPUS_STR = str(os.environ.get("NOTEBOOK_OCPUS", os.cpu_count() or 4))
os.environ["OMP_NUM_THREADS"] = NCPUS_STR
os.environ["OPENBLAS_NUM_THREADS"] = NCPUS_STR
os.environ["MKL_NUM_THREADS"] = NCPUS_STR

import sys
import json
import time
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import category_encoders as ce

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------------
# Importar config e utils do projeto
# ---------------------------------------------------------------------------
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from config import *
from utils import *

print(f"CPUs: {NCPUS} | Target: {TARGET}")
print(f"Features: {len(SELECTED_FEATURES)} ({len(NUM_FEATURES)} num + {len(CAT_FEATURES)} cat)")
print(f"Train SAFRAs: {TRAIN_SAFRAS} | OOS: {OOS_SAFRA} | OOT: {OOT_SAFRAS}")

In [ ]:
# ---------------------------------------------------------------------------
# Carregar feature store da Gold
# ---------------------------------------------------------------------------
log("Carregando feature store...")

cols_to_load = ["NUM_CPF", "SAFRA", TARGET] + SELECTED_FEATURES
df = pd.read_parquet(LOCAL_DATA_PATH, columns=cols_to_load, engine="pyarrow")

log(f"Shape: {df.shape}")
log(f"Taxa FPD: {df[TARGET].mean():.4f} ({df[TARGET].sum():,} / {len(df):,})")
log(f"SAFRAs: {sorted(df['SAFRA'].unique())}")

In [ ]:
# ---------------------------------------------------------------------------
# Split temporal: Train / OOS / OOT
# ---------------------------------------------------------------------------
def temporal_split(df, features):
    """Split por SAFRA em train, OOS e OOT."""
    mask_train = df["SAFRA"].isin(TRAIN_SAFRAS)
    mask_oos = df["SAFRA"].isin(OOS_SAFRA)
    mask_oot = df["SAFRA"].isin(OOT_SAFRAS)

    X_train = df.loc[mask_train, features].copy()
    y_train = df.loc[mask_train, TARGET].values
    X_oos = df.loc[mask_oos, features].copy()
    y_oos = df.loc[mask_oos, TARGET].values
    X_oot = df.loc[mask_oot, features].copy()
    y_oot = df.loc[mask_oot, TARGET].values

    log(f"Train: {X_train.shape} (FPD={y_train.mean():.4f})")
    log(f"OOS:   {X_oos.shape} (FPD={y_oos.mean():.4f})")
    log(f"OOT:   {X_oot.shape} (FPD={y_oot.mean():.4f})")

    return X_train, y_train, X_oos, y_oos, X_oot, y_oot


X_train, y_train, X_oos, y_oos, X_oot, y_oot = temporal_split(df, SELECTED_FEATURES)

In [ ]:
# ---------------------------------------------------------------------------
# Preprocessing pipeline
# ---------------------------------------------------------------------------
def build_prep(use_scaler=False):
    """Constroi ColumnTransformer com imputacao e encoding."""
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if use_scaler:
        num_steps.append(("scaler", StandardScaler()))
    num_pipeline = Pipeline(num_steps)

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", ce.CountEncoder(handle_unknown=0, handle_missing="count")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipeline, NUM_FEATURES),
            ("cat", cat_pipeline, CAT_FEATURES),
        ],
        remainder="drop",
    )
    return preprocessor


log("build_prep() definido.")

In [ ]:
# ---------------------------------------------------------------------------
# Funcao de avaliacao de modelo
# ---------------------------------------------------------------------------
all_metrics = []  # acumulador global


def eval_model(pipe, name, X_tr, y_tr, X_os, y_os, X_ot, y_ot):
    """Avalia modelo em train/OOS/OOT. Retorna dict de metricas."""
    metrics = {"model": name}

    for split_name, X, y in [("train", X_tr, y_tr), ("oos", X_os, y_os), ("oot", X_ot, y_ot)]:
        y_prob = pipe.predict_proba(X)[:, 1]
        auc = roc_auc_score(y, y_prob)
        ks = compute_ks(y, y_prob)
        gini = compute_gini(auc)
        metrics[f"ks_{split_name}"] = round(ks, 4)
        metrics[f"auc_{split_name}"] = round(auc, 4)
        metrics[f"gini_{split_name}"] = round(gini, 2)

    # PSI: train vs OOT scores
    prob_train = pipe.predict_proba(X_tr)[:, 1]
    prob_oot = pipe.predict_proba(X_ot)[:, 1]
    metrics["psi"] = compute_psi(prob_train, prob_oot)
    metrics["psi_alert"] = psi_alert(metrics["psi"])

    all_metrics.append(metrics)
    log(f"{name}: KS_train={metrics['ks_train']:.4f} | KS_oos={metrics['ks_oos']:.4f} | "
        f"KS_oot={metrics['ks_oot']:.4f} | AUC_oot={metrics['auc_oot']:.4f} | PSI={metrics['psi']:.4f}")

    return metrics


log("eval_model() definido.")

## HPO com Optuna (50 trials)

Otimizacao de hiperparametros usando **Optuna** com amostragem TPE (Tree-structured
Parzen Estimators) e validacao cruzada 3-fold estratificada.

Modelos otimizados: LightGBM, XGBoost, CatBoost.

A flag `SKIP_HPO` permite pular a otimizacao e carregar parametros pre-calculados
de `artifacts/hpo/best_params_all.json`.

In [ ]:
# ---------------------------------------------------------------------------
# HPO com Optuna — LightGBM / XGBoost / CatBoost
# ---------------------------------------------------------------------------
def _run_hpo_model(df_hpo, features, model_type, n_trials=50):
    """Executa HPO para um modelo especifico com 3-fold StratifiedKFold."""
    log(f"HPO: {model_type} ({n_trials} trials)...")

    X_hpo = df_hpo[features].copy()
    y_hpo = df_hpo[TARGET].values

    # Preprocessing
    prep = build_prep(use_scaler=False)
    X_hpo_transformed = prep.fit_transform(X_hpo)

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    def objective(trial):
        if model_type == "lightgbm":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 200, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 8),
                "num_leaves": trial.suggest_int("num_leaves", 15, 127),
                "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "random_state": 42, "verbose": -1, "n_jobs": NCPUS,
                "is_unbalance": True,
            }
            model = lgb.LGBMClassifier(**params)

        elif model_type == "xgboost":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 200, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 8),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "random_state": 42, "verbosity": 0, "n_jobs": NCPUS,
                "scale_pos_weight": (y_hpo == 0).sum() / max((y_hpo == 1).sum(), 1),
                "eval_metric": "auc", "use_label_encoder": False,
            }
            model = xgb.XGBClassifier(**params)

        elif model_type == "catboost":
            params = {
                "iterations": trial.suggest_int("iterations", 200, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "depth": trial.suggest_int("depth", 3, 8),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
                "random_seed": 42, "verbose": 0, "thread_count": NCPUS,
                "auto_class_weights": "Balanced",
            }
            model = CatBoostClassifier(**params)
        else:
            raise ValueError(f"Modelo nao suportado: {model_type}")

        # 3-fold CV
        aucs = []
        for train_idx, val_idx in skf.split(X_hpo_transformed, y_hpo):
            X_fold_train = X_hpo_transformed[train_idx]
            y_fold_train = y_hpo[train_idx]
            X_fold_val = X_hpo_transformed[val_idx]
            y_fold_val = y_hpo[val_idx]

            model.fit(X_fold_train, y_fold_train)
            y_prob = model.predict_proba(X_fold_val)[:, 1]
            aucs.append(roc_auc_score(y_fold_val, y_prob))

        return np.mean(aucs)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    log(f"HPO {model_type} concluido. Melhor AUC CV: {study.best_value:.4f}")
    return study.best_params


log("_run_hpo_model() definido.")

In [ ]:
# ---------------------------------------------------------------------------
# Executar HPO ou carregar parametros salvos
# ---------------------------------------------------------------------------
SKIP_HPO = True  # Alterar para False para rodar HPO completo

hpo_dir = os.path.join(ARTIFACT_DIR, "hpo")
hpo_path = os.path.join(hpo_dir, "best_params_all.json")

if SKIP_HPO and os.path.exists(hpo_path):
    log(f"Carregando parametros HPO de {hpo_path}...")
    with open(hpo_path, "r") as f:
        best_params_all = json.load(f)
    for model_name, params in best_params_all.items():
        log(f"  {model_name}: {params}")
else:
    if SKIP_HPO:
        log("SKIP_HPO=True mas arquivo nao encontrado. Usando defaults.")
        best_params_all = {
            "lightgbm": {
                "n_estimators": 500, "learning_rate": 0.03, "max_depth": 6,
                "num_leaves": 63, "min_child_samples": 50, "subsample": 0.8,
                "colsample_bytree": 0.8, "reg_alpha": 0.1, "reg_lambda": 1.0,
            },
            "xgboost": {
                "n_estimators": 500, "learning_rate": 0.03, "max_depth": 6,
                "min_child_weight": 10, "subsample": 0.8,
                "colsample_bytree": 0.8, "reg_alpha": 0.1, "reg_lambda": 1.0,
            },
            "catboost": {
                "iterations": 500, "learning_rate": 0.03, "depth": 6,
                "l2_leaf_reg": 3.0, "bagging_temperature": 0.5, "random_strength": 1.0,
            },
        }
    else:
        log("Executando HPO completo (50 trials por modelo)...")
        df_train_hpo = df[df["SAFRA"].isin(TRAIN_SAFRAS)]
        best_params_all = {}
        for m_type in ["lightgbm", "xgboost", "catboost"]:
            t0 = time.time()
            best_params_all[m_type] = _run_hpo_model(df_train_hpo, SELECTED_FEATURES, m_type, n_trials=50)
            log(f"  {m_type} HPO: {time.time() - t0:.0f}s")

        # Salvar
        os.makedirs(hpo_dir, exist_ok=True)
        with open(hpo_path, "w") as f:
            json.dump(best_params_all, f, indent=2)
        log(f"Parametros HPO salvos em {hpo_path}")

print(f"\nParametros carregados para: {list(best_params_all.keys())}")

## Treinamento: 5 Modelos

Treinamento final de cada modelo com os melhores hiperparametros:

1. **LightGBM v2** — Gradient boosting com HPO
2. **XGBoost** — Gradient boosting com HPO
3. **CatBoost** — Gradient boosting com HPO
4. **Random Forest** — Ensemble com parametros fixos
5. **Logistic Regression L1 v2** — Modelo linear com StandardScaler

In [ ]:
# ---------------------------------------------------------------------------
# Modelo 1: LightGBM v2
# ---------------------------------------------------------------------------
log("Treinando LightGBM v2...")
t0 = time.time()

lgb_params = best_params_all.get("lightgbm", {})
lgb_model = lgb.LGBMClassifier(
    n_estimators=lgb_params.get("n_estimators", 500),
    learning_rate=lgb_params.get("learning_rate", 0.03),
    max_depth=lgb_params.get("max_depth", 6),
    num_leaves=lgb_params.get("num_leaves", 63),
    min_child_samples=lgb_params.get("min_child_samples", 50),
    subsample=lgb_params.get("subsample", 0.8),
    colsample_bytree=lgb_params.get("colsample_bytree", 0.8),
    reg_alpha=lgb_params.get("reg_alpha", 0.1),
    reg_lambda=lgb_params.get("reg_lambda", 1.0),
    is_unbalance=True,
    random_state=42,
    verbose=-1,
    n_jobs=NCPUS,
)

pipe_lgb = Pipeline([
    ("prep", build_prep(use_scaler=False)),
    ("model", lgb_model),
])
pipe_lgb.fit(X_train, y_train)
metrics_lgb = eval_model(pipe_lgb, "LightGBM_v2", X_train, y_train, X_oos, y_oos, X_oot, y_oot)
log(f"LightGBM v2 treinado em {time.time() - t0:.1f}s")

In [ ]:
# ---------------------------------------------------------------------------
# Modelo 2: XGBoost
# ---------------------------------------------------------------------------
log("Treinando XGBoost...")
t0 = time.time()

xgb_params = best_params_all.get("xgboost", {})
xgb_model = xgb.XGBClassifier(
    n_estimators=xgb_params.get("n_estimators", 500),
    learning_rate=xgb_params.get("learning_rate", 0.03),
    max_depth=xgb_params.get("max_depth", 6),
    min_child_weight=xgb_params.get("min_child_weight", 10),
    subsample=xgb_params.get("subsample", 0.8),
    colsample_bytree=xgb_params.get("colsample_bytree", 0.8),
    reg_alpha=xgb_params.get("reg_alpha", 0.1),
    reg_lambda=xgb_params.get("reg_lambda", 1.0),
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42,
    verbosity=0,
    n_jobs=NCPUS,
)

pipe_xgb = Pipeline([
    ("prep", build_prep(use_scaler=False)),
    ("model", xgb_model),
])
pipe_xgb.fit(X_train, y_train)
metrics_xgb = eval_model(pipe_xgb, "XGBoost", X_train, y_train, X_oos, y_oos, X_oot, y_oot)
log(f"XGBoost treinado em {time.time() - t0:.1f}s")

In [ ]:
# ---------------------------------------------------------------------------
# Modelo 3: CatBoost
# ---------------------------------------------------------------------------
log("Treinando CatBoost...")
t0 = time.time()

cb_params = best_params_all.get("catboost", {})
cb_model = CatBoostClassifier(
    iterations=cb_params.get("iterations", 500),
    learning_rate=cb_params.get("learning_rate", 0.03),
    depth=cb_params.get("depth", 6),
    l2_leaf_reg=cb_params.get("l2_leaf_reg", 3.0),
    bagging_temperature=cb_params.get("bagging_temperature", 0.5),
    random_strength=cb_params.get("random_strength", 1.0),
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=0,
    thread_count=NCPUS,
)

pipe_cb = Pipeline([
    ("prep", build_prep(use_scaler=False)),
    ("model", cb_model),
])
pipe_cb.fit(X_train, y_train)
metrics_cb = eval_model(pipe_cb, "CatBoost", X_train, y_train, X_oos, y_oos, X_oot, y_oot)
log(f"CatBoost treinado em {time.time() - t0:.1f}s")

In [ ]:
# ---------------------------------------------------------------------------
# Modelo 4: Random Forest
# ---------------------------------------------------------------------------
log("Treinando Random Forest...")
t0 = time.time()

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=NCPUS,
)

pipe_rf = Pipeline([
    ("prep", build_prep(use_scaler=False)),
    ("model", rf_model),
])
pipe_rf.fit(X_train, y_train)
metrics_rf = eval_model(pipe_rf, "RandomForest", X_train, y_train, X_oos, y_oos, X_oot, y_oot)
log(f"Random Forest treinado em {time.time() - t0:.1f}s")

In [ ]:
# ---------------------------------------------------------------------------
# Modelo 5: Logistic Regression L1 v2 (com StandardScaler)
# ---------------------------------------------------------------------------
log("Treinando Logistic Regression L1 v2...")
t0 = time.time()

lr_model = LogisticRegression(
    penalty="l1",
    C=0.5,
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)

pipe_lr = Pipeline([
    ("prep", build_prep(use_scaler=True)),  # StandardScaler ativado
    ("model", lr_model),
])
pipe_lr.fit(X_train, y_train)
metrics_lr = eval_model(pipe_lr, "LR_L1_v2", X_train, y_train, X_oos, y_oos, X_oot, y_oot)
log(f"LR L1 v2 treinado em {time.time() - t0:.1f}s")

## Avaliacao Comparativa

Comparacao lado a lado dos 5 modelos em todas as metricas e splits.

In [ ]:
# ---------------------------------------------------------------------------
# Tabela comparativa
# ---------------------------------------------------------------------------
metrics_df = pd.DataFrame(all_metrics)
display_cols = [
    "model",
    "ks_train", "ks_oos", "ks_oot",
    "auc_train", "auc_oos", "auc_oot",
    "gini_oot",
    "psi", "psi_alert",
]

print("=" * 100)
print("COMPARACAO DE MODELOS")
print("=" * 100)
print(metrics_df[display_cols].to_string(index=False))
print("=" * 100)

# Melhor modelo por KS OOT
best = metrics_df.loc[metrics_df["ks_oot"].idxmax()]
print(f"\nMelhor modelo (KS OOT): {best['model']} — KS={best['ks_oot']:.4f}, AUC={best['auc_oot']:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Plots: ROC, KS, score distributions, feature importance
# ---------------------------------------------------------------------------
plots_dir = os.path.join(ARTIFACT_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

models_dict = {
    "LightGBM_v2": pipe_lgb,
    "XGBoost": pipe_xgb,
    "CatBoost": pipe_cb,
    "RandomForest": pipe_rf,
    "LR_L1_v2": pipe_lr,
}

# --- Plot 1: ROC curves overlay (OOT) ---
fig, ax = plt.subplots(figsize=(10, 8))
for name, pipe in models_dict.items():
    y_prob = pipe.predict_proba(X_oot)[:, 1]
    fpr, tpr, _ = roc_curve(y_oot, y_prob)
    auc_val = roc_auc_score(y_oot, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})", linewidth=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curvas ROC — OOT (202502-202503)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "roc_curves_oot.png"), dpi=150)
plt.show()

# --- Plot 2: KS curves (OOT - melhor modelo) ---
fig, ax = plt.subplots(figsize=(10, 8))
best_name = best["model"]
best_pipe = models_dict[best_name]
y_prob_best = best_pipe.predict_proba(X_oot)[:, 1]

# KS curve
thresholds = np.linspace(0, 1, 200)
tpr_list, fpr_list = [], []
for t in thresholds:
    tpr_list.append((y_prob_best[y_oot == 1] >= t).mean())
    fpr_list.append((y_prob_best[y_oot == 0] >= t).mean())
tpr_arr = np.array(tpr_list)
fpr_arr = np.array(fpr_list)
ks_curve = tpr_arr - fpr_arr
ks_max_idx = np.argmax(ks_curve)

ax.plot(thresholds, tpr_arr, label="TPR (Maus)", linewidth=2)
ax.plot(thresholds, fpr_arr, label="FPR (Bons)", linewidth=2)
ax.fill_between(thresholds, fpr_arr, tpr_arr, alpha=0.15, color="green")
ax.axvline(x=thresholds[ks_max_idx], color="red", linestyle="--",
           label=f"KS max = {ks_curve[ks_max_idx]:.4f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Taxa Cumulativa")
ax.set_title(f"Curva KS — {best_name} (OOT)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "ks_curve_best.png"), dpi=150)
plt.show()

# --- Plot 3: Score distributions (OOT - melhor modelo) ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(y_prob_best[y_oot == 0], bins=50, alpha=0.6, label="Bons (FPD=0)", density=True, color="#3498db")
ax.hist(y_prob_best[y_oot == 1], bins=50, alpha=0.6, label="Maus (FPD=1)", density=True, color="#e74c3c")
ax.set_xlabel("Score (P(FPD=1))")
ax.set_ylabel("Densidade")
ax.set_title(f"Distribuicao de Scores — {best_name} (OOT)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "score_distribution_oot.png"), dpi=150)
plt.show()

# --- Plot 4: Feature importance (top 20 LightGBM) ---
fig, ax = plt.subplots(figsize=(10, 8))
lgb_importances = pipe_lgb.named_steps["model"].feature_importances_
# Reconstruir nomes das features apos ColumnTransformer
feature_names = NUM_FEATURES + CAT_FEATURES
imp_df = pd.DataFrame({"feature": feature_names, "importance": lgb_importances})
imp_df = imp_df.sort_values("importance", ascending=False).head(20)

ax.barh(imp_df["feature"][::-1], imp_df["importance"][::-1], color="#2ecc71")
ax.set_xlabel("Importance (split count)")
ax.set_title("Top 20 Features — LightGBM v2")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "feature_importance_lgb.png"), dpi=150)
plt.show()

log(f"Todos os plots salvos em {plots_dir}")

In [ ]:
# ---------------------------------------------------------------------------
# Salvar modelos (.pkl) e metricas (.json)
# ---------------------------------------------------------------------------
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Salvar pipelines completos
for name, pipe in models_dict.items():
    model_path = os.path.join(ARTIFACT_DIR, f"{name}.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(pipe, f)
    log(f"Modelo salvo: {model_path}")

# Salvar metricas consolidadas
metrics_path = os.path.join(ARTIFACT_DIR, "model_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
log(f"Metricas salvas: {metrics_path}")

# Resumo final
print(f"\n{'=' * 70}")
print(f"TREINAMENTO CONCLUIDO")
print(f"{'=' * 70}")
print(f"Modelos treinados: {len(models_dict)}")
print(f"Melhor modelo (KS OOT): {best['model']}")
print(f"  KS OOT:  {best['ks_oot']:.4f}")
print(f"  AUC OOT: {best['auc_oot']:.4f}")
print(f"  Gini:    {best['gini_oot']:.2f}")
print(f"  PSI:     {best['psi']:.4f} ({best['psi_alert']})")
print(f"Artefatos em: {ARTIFACT_DIR}")
print(f"{'=' * 70}")